# PneumoScan: train ResNet50 and the custom CNN on a free Colab GPU

**Before you start (on your Mac, once)**

1. Zip the backend *without* the virtual environments, old models and your `.env` (it holds your API key):
```bash
cd ~/Desktop/pneumoscan
zip -rq pneumoscan_colab.zip backend \
  -x "backend/venv/*" "backend/.venv/*" "backend/ml_env/*" \
     "backend/saved_models/*" "backend/outputs/*" "backend/.env" \
     "*/__pycache__/*" "*.DS_Store"
```
2. Upload `pneumoscan_colab.zip` to the **top level of My Drive**.
3. In Colab choose **Runtime > Change runtime type > T4 GPU**.
4. Run the cells below **one at a time, in order** (do not use "Run all").

Your own copy of the dataset keeps the split identical to your MobileNetV2 run, so the comparison is fair.

## 1. Check the GPU

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "No GPU found")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then run this cell again."
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Copy the project from Drive and unzip it

In [ ]:
import os, shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")

ZIP_ON_DRIVE = "/content/drive/MyDrive/pneumoscan_colab.zip"
assert os.path.exists(ZIP_ON_DRIVE), f"{ZIP_ON_DRIVE} not found. Upload the zip to the top level of My Drive."

shutil.copy(ZIP_ON_DRIVE, "/content/pneumoscan_colab.zip")      # local disk is much faster than Drive
os.makedirs("/content/pneumoscan", exist_ok=True)
subprocess.run(["unzip", "-q", "-o", "/content/pneumoscan_colab.zip", "-d", "/content/pneumoscan"], check=True)
shutil.rmtree("/content/pneumoscan/__MACOSX", ignore_errors=True)
print(os.listdir("/content/pneumoscan/backend"))

## 3. Install two small packages and check the dataset

In [ ]:
%pip install -q python-dotenv certifi

In [ ]:
import os

BACKEND = "/content/pneumoscan/backend"
DATA = f"{BACKEND}/data/chest_xray"
exts = (".jpg", ".jpeg", ".png")

counts = {}
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        d = f"{DATA}/{split}/{cls}"
        counts[(split, cls)] = len([f for f in os.listdir(d) if f.lower().endswith(exts)]) if os.path.isdir(d) else 0
        print(f"{split:5s} {cls:10s} {counts[(split, cls)]}")

assert counts[("train", "NORMAL")] > 0 and counts[("train", "PNEUMONIA")] > 0, "Training images not found. Check the zip contents."
assert counts[("test", "NORMAL")] > 0 and counts[("test", "PNEUMONIA")] > 0, "Test images not found. Check the zip contents."
print("\nThe Kaggle test set should show 234 NORMAL and 390 PNEUMONIA.")

## 4. Helper: back up each finished model to Drive

Run this once. It defines a function that the training cells call, so a finished model is safe on Drive
even if Colab disconnects before the next one.

In [ ]:
import os, shutil

BACKEND = "/content/pneumoscan/backend"
DRIVE_OUT = "/content/drive/MyDrive/pneumoscan_outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)

def save_to_drive(run_name):
    files_ = [
        f"{BACKEND}/saved_models/{run_name}.pt",
        f"{BACKEND}/outputs/history_{run_name}.json",
        f"{BACKEND}/outputs/plots/curves_{run_name}.png",
    ]
    for f in files_:
        assert os.path.exists(f), f"Missing {f}. Did training finish?"
        shutil.copy(f, DRIVE_OUT)
    print("Backed up to Drive:", DRIVE_OUT, "->", run_name)

## 5. Train ResNet50

The output must start with `Device: cuda`. Wait for `Best val macro-F1 ... Saved -> ...` before moving on.

In [ ]:
%cd /content/pneumoscan/backend
!python train.py --model resnet50 --loss weighted_ce --epochs 10
save_to_drive("resnet50_weighted_ce")

## 6. Train the custom CNN (from scratch)

It starts from random weights, so it needs more epochs. Early stopping ends it sooner if it stops improving.

In [ ]:
%cd /content/pneumoscan/backend
!python train.py --model custom_cnn --loss weighted_ce --epochs 25
save_to_drive("custom_cnn_weighted_ce")

## 7. Check both results

In [ ]:
import os, torch
from IPython.display import Image, display

%cd /content/pneumoscan/backend
for run, expected in [("resnet50_weighted_ce", "resnet50"), ("custom_cnn_weighted_ce", "custom_cnn")]:
    path = f"saved_models/{run}.pt"
    if not os.path.exists(path):
        print(f"{run}: not trained yet, skipping")
        continue
    # weights_only=False: this is your own checkpoint (it stores plain Python values next to the weights)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    assert ckpt["model_name"] == expected and "state" in ckpt
    print(f"{run}: best epoch {ckpt['epoch']} | best val macro-F1 {float(ckpt['best_val_f1']):.4f}")
    display(Image(f"outputs/plots/curves_{run}.png"))

## 8. Download everything as one zip

The zip contains the model files, histories and curves in the same folder layout as `backend/`,
so on your Mac one command puts them in the right place.
**Do not copy `metrics.json` or ROC/confusion plots from Colab**: run `python evaluate.py` on your Mac instead.

In [ ]:
import os, zipfile
from google.colab import files

%cd /content/pneumoscan/backend
runs = [r for r in ["resnet50_weighted_ce", "custom_cnn_weighted_ce"] if os.path.exists(f"saved_models/{r}.pt")]
assert runs, "No trained models found. Run the training cells first."

out_zip = "/content/pneumoscan_models.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for r in runs:
        for f in [f"saved_models/{r}.pt", f"outputs/history_{r}.json", f"outputs/plots/curves_{r}.png"]:
            z.write(f, f)
            print("added", f)

import shutil
shutil.copy(out_zip, "/content/drive/MyDrive/pneumoscan_outputs/")   # extra backup on Drive
files.download(out_zip)

## 9. Back on your Mac

Run these in Terminal (the zip goes into your Downloads folder):
```bash
unzip -o ~/Downloads/pneumoscan_models.zip -d ~/Desktop/pneumoscan/backend
ls -lh ~/Desktop/pneumoscan/backend/saved_models

cd ~/Desktop/pneumoscan/backend
source ../pytorch_env/bin/activate
python evaluate.py
python compare_models.py
```
`evaluate.py` evaluates every model in `saved_models/` (MobileNetV2, ResNet50 and the custom CNN) and `compare_models.py` builds the comparison table and chart.